# Per-make/model pricing notebook

A single parameterised view of one make/model:
what the asking-price cloud looks like, what a given (year, mileage) query is
worth as a **realistic sale price**, and what the accrued price-history /
lifecycle signals say about cuts and days-on-market.

All estimation lives in `analysis.py`; this notebook only *reads* the DB and
draws. Change the parameters in the next cell and **Run All**.

## Parameters

Edit these, then Run All. `MODEL = None` pools every model of `MAKE`.

In [34]:
# --- Query parameters (edit me) ---------------------------------------------
MAKE = "mazda"          # URL slug or display name; matching is case/punctuation-insensitive
MODEL = "cx-5"         # e.g. "cx-30"; set to None to pool all models of MAKE

# The (year, mileage) box you are considering importing/bidding on.
QUERY_YEAR_RANGE = (2023, 2023)        # inclusive calendar years, or None to use the data span
QUERY_MILEAGE_RANGE = (55_000, 65_000) # km, or None to use the data span
FUEL_TYPE = "Petrol"                   # "Petrol" / "Diesel" / ...; None = pool every fuel
EXCLUDE_AVAILABILITY = ("In transit",) # advert availabilities to drop; () keeps every advert

# Fuel is a *hard subset filter*: every other fuel is dropped before anything is
# computed, so the fit, the comparables, the price-cut and days-on-market signals
# and all six charts see this fuel alone. Listings whose fuel is unknown go with
# them -- they cannot be shown to be the fuel asked for. Set FUEL_TYPE = None to
# pool every fuel instead; the curve then prices fuel through a dummy coefficient
# and fuels thinner than analysis.MIN_LEVEL_COUNT fall back to the baseline fuel.

# EXCLUDE_AVAILABILITY drops adverts by the site's "Availability" field before
# anything else runs. "In transit" cars are not on the island yet: their asking
# price is an import quote nobody can view or bid on today, so they would bias
# the fit and the comparables. Availability-unknown adverts are kept -- they
# cannot be shown to be in transit. Set EXCLUDE_AVAILABILITY = () to keep them all.


In [35]:
# analysis.py is edited alongside this notebook; reload it on every cell run
# so a change there does not need a kernel restart.
%load_ext autoreload
%autoreload 2

import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

from bazaraki import db
from bazaraki import analysis

warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

EUR = FuncFormatter(lambda v, _pos: f"\u20ac{v:,.0f}")
KM = FuncFormatter(lambda v, _pos: f"{v/1000:.0f}k")
MODEL_LABEL = f"{MAKE} {MODEL}" if MODEL else f"{MAKE} (all models)"
if FUEL_TYPE:
    MODEL_LABEL += f" ({FUEL_TYPE.lower()})"
print("Analysing:", MODEL_LABEL)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Analysing: mazda cx-5 (petrol)


## Load & scope the data

One tidy frame per make/model — restricted to `FUEL_TYPE`, which is excluded
from the data rather than modelled, and with `EXCLUDE_AVAILABILITY` adverts
("In transit") dropped outright — plus the *cleaned* subset the model
actually fits on (`analysis.is_usable` drops nulls and physical
impossibilities), the fitted hedonic curve, and each advert's price trajectory.

In [36]:
# All listings for this make/model, before the availability and fuel filters.
every_record = analysis.filter_model(analysis.to_records(db.all_listings()), MAKE, MODEL)

# Availability first: "In transit" adverts are import quotes for cars that are
# not here yet, so they are dropped from the market entirely (see the
# parameters cell) before the fuel mix below is even counted.
all_records = analysis.exclude_availability(every_record, *EXCLUDE_AVAILABILITY)

# FUEL_TYPE then excludes the other fuels outright, so everything downstream --
# frame, fit, comparables, histories, charts -- is computed on this fuel only.
# FUEL_TYPE = None keeps every fuel.
records = analysis.filter_fuel(all_records, FUEL_TYPE)

rows = [{
    "ad_id": r.ad_id, "price": r.price, "year": r.year, "mileage_km": r.mileage_km,
    "fuel_type": r.fuel_type, "gearbox": r.gearbox, "seller_type": r.seller_type,
    "availability": r.availability,
    "is_active": r.is_active, "days_on_market": r.days_on_market,
} for r in records]
df = pd.DataFrame(rows)

# The subset used for fitting: has price/year/mileage and passes plausibility.
usable = analysis.clean(records)
df_clean = df[df.ad_id.isin({r.ad_id for r in usable})].copy()

# Fit the hedonic asking-price curve once; reused by several charts.
curve = analysis.fit_price_curve(usable)

# Canonical spelling of the fuel as the data writes it, for predict() below.
FUEL = analysis._canonical_level(usable, "fuel_type", FUEL_TYPE)

# Price trajectories (oldest-first) keyed by advert, for the cuts chart.
histories = {
    r.ad_id: [(o.observed_at, o.price) for o in db.price_history(r.ad_id)]
    for r in records
}

print(f"{len(df)} listings  |  {len(df_clean)} usable for the fit  |  "
      f"{int(df.is_active.sum())} active / {int((~df.is_active).sum())} delisted")
print("availability mix before the filter:",
      pd.Series([r.availability for r in every_record]).value_counts(dropna=False).to_dict())
if EXCLUDE_AVAILABILITY:
    print(f"availability filter {list(EXCLUDE_AVAILABILITY)}: kept {len(all_records)} of "
          f"{len(every_record)} listings, {len(every_record) - len(all_records)} excluded")
print("fuel mix after it:",
      pd.Series([r.fuel_type for r in all_records]).value_counts(dropna=False).to_dict())
if FUEL_TYPE:
    print(f"fuel filter {FUEL_TYPE!r}: kept {len(records)} of {len(all_records)} listings, "
          f"{len(all_records) - len(records)} excluded")
elif curve is not None:
    # No filter: fuel is priced inside the curve instead of being excluded.
    fit_levels = curve.spec.categoricals.get("fuel_type")
    print("fuel: not a factor in the fit \u2014 every fuel shares one curve" if fit_levels is None
          else f"fuel: baseline {fit_levels[0]!r}, priced apart from " + ", ".join(fit_levels[1]))
if curve is None:
    print("WARNING: too few usable rows to fit a curve \u2014 charts 1\u20133/5 will be limited.")


TypeError: CarRecord.__init__() got an unexpected keyword argument 'availability'

## The query answer

The actionable triple — realistic sale price, a range, expected days-on-market — plus N and confidence.

In [ ]:
# analysis.estimate_from_db would re-read every fuel from the DB, so feed the
# same estimator the fuel-filtered records (and their price trajectories) instead.
est = analysis.estimate_sale_price(
    records, MAKE, MODEL, QUERY_YEAR_RANGE, QUERY_MILEAGE_RANGE, FUEL_TYPE,
    histories=[[p for _, p in histories[r.ad_id]] for r in records],
)

def eur(x):
    return "n/a" if x is None else f"\u20ac{x:,.0f}"

print(f"Query: {MODEL_LABEL}, years {QUERY_YEAR_RANGE}, mileage {QUERY_MILEAGE_RANGE} km, fuel {FUEL_TYPE or 'all pooled'}\n")
print(f"  Realistic sale price : {eur(est.sale_price)}")
print(f"  Range (P25\u2013P75)      : {eur(est.range_low)} \u2013 {eur(est.range_high)}")
print(f"  Asking-curve estimate: {eur(est.asking_estimate)}  (\u00d7{est.adjustment_factor:.3f} asking\u2192sale)")
print(f"  Comparables median   : {eur(est.comparables_median)}")
dom = est.expected_days_on_market
print(f"  Expected days on mkt : {dom if dom is not None else 'n/a (not enough delisted comparables)'}")
print(f"  Sample size / conf.  : N={est.n}, {est.confidence}")


## 1 · Price vs. mileage

Scatter coloured by model year, with the fitted regression curve (held at the
median year) and its prediction-interval band.

In [ ]:
fig, ax = plt.subplots()
if not df_clean.empty:
    sc = ax.scatter(df_clean.mileage_km, df_clean.price, c=df_clean.year,
                    cmap="viridis", s=36, alpha=0.8, edgecolor="white", linewidth=0.4)
    fig.colorbar(sc, ax=ax, label="Model year")

    if curve is not None:
        ref_year = int(round(df_clean.year.median()))
        grid = np.linspace(df_clean.mileage_km.min(), df_clean.mileage_km.max(), 80)
        preds = [analysis.predict(curve, ref_year, int(m), fuel_type=FUEL, alpha=0.05) for m in grid]
        mean = [p.estimate for p in preds]
        lo = [p.lo for p in preds]
        hi = [p.hi for p in preds]
        ax.plot(grid, mean, color="crimson", lw=2, label=f"Fit @ {ref_year}")
        if all(v is not None for v in lo + hi):
            ax.fill_between(grid, lo, hi, color="crimson", alpha=0.12, label="95% interval")
        ax.legend()

ax.set_xlabel("Mileage")
ax.set_ylabel("Asking price")
ax.xaxis.set_major_formatter(KM)
ax.yaxis.set_major_formatter(EUR)
ax.set_title(f"{MODEL_LABEL} \u2014 price vs. mileage")
plt.show()


## 2 · Price vs. year (depreciation)

The same cloud against model year, with the fitted curve held at the median
mileage — the depreciation slope the hedonic model implies.

In [ ]:
fig, ax = plt.subplots()
if not df_clean.empty:
    ax.scatter(df_clean.year, df_clean.price, c=df_clean.mileage_km, cmap="magma_r",
               s=36, alpha=0.8, edgecolor="white", linewidth=0.4)
    cb = fig.colorbar(plt.cm.ScalarMappable(
        norm=plt.Normalize(df_clean.mileage_km.min(), df_clean.mileage_km.max()),
        cmap="magma_r"), ax=ax)
    cb.set_label("Mileage (km)")

    if curve is not None:
        ref_km = int(round(df_clean.mileage_km.median()))
        years = np.arange(int(df_clean.year.min()), int(df_clean.year.max()) + 1)
        mean = [analysis.predict(curve, int(y), ref_km, fuel_type=FUEL, alpha=0.05).estimate
                for y in years]
        ax.plot(years, mean, color="crimson", lw=2, marker="o", ms=4,
                label=f"Fit @ {ref_km/1000:.0f}k km")
        ax.legend()

ax.set_xlabel("Model year")
ax.set_ylabel("Asking price")
ax.yaxis.set_major_formatter(EUR)
ax.set_title(f"{MODEL_LABEL} \u2014 depreciation by year")
plt.show()


## 3 · Your query against the cloud

The price-vs-mileage cloud again, with the query's year/mileage box shaded and
the estimated **sale price** (with its P25–P75 range) marked at the box centre.

In [ ]:
fig, ax = plt.subplots()
if not df_clean.empty:
    ax.scatter(df_clean.mileage_km, df_clean.price, s=30, alpha=0.35,
               color="steelblue", edgecolor="none", label="listings")

# Resolve the query box (fall back to the data span, mirroring analysis.py).
yr = QUERY_YEAR_RANGE or (int(df_clean.year.min()), int(df_clean.year.max()))
mr = QUERY_MILEAGE_RANGE or (int(df_clean.mileage_km.min()), int(df_clean.mileage_km.max()))
mid_km = sum(mr) / 2

ax.axvspan(mr[0], mr[1], color="orange", alpha=0.10,
           label=f"query mileage {mr[0]/1000:.0f}\u2013{mr[1]/1000:.0f}k")

if est.sale_price is not None:
    yerr = None
    if est.range_low is not None and est.range_high is not None:
        yerr = [[est.sale_price - est.range_low], [est.range_high - est.sale_price]]
    ax.errorbar([mid_km], [est.sale_price], yerr=yerr, fmt="D", color="crimson",
                ms=11, capsize=6, lw=2, zorder=5,
                label=f"sale est. \u2248 \u20ac{est.sale_price:,.0f}")
    if est.asking_estimate is not None:
        ax.scatter([mid_km], [est.asking_estimate], marker="_", s=400,
                   color="black", zorder=6, label="asking-curve est.")

ax.set_xlabel("Mileage")
ax.set_ylabel("Price")
ax.xaxis.set_major_formatter(KM)
ax.yaxis.set_major_formatter(EUR)
ax.set_title(f"{MODEL_LABEL} \u2014 query {yr[0]}\u2013{yr[1]} vs. the market")
ax.legend()
plt.show()


## 4 · Price history & cuts

Left: the price trajectory of every advert that changed price (asking prices
drift down toward a sale). Right: the distribution of first→last cut %, the raw
material for the Layer-2 asking→sale haircut. Needs accrued daily runs — sparse
until history builds up.

In [ ]:
changed = {aid: h for aid, h in histories.items() if len({p for _, p in h}) > 1}
cut_signal = analysis.price_cut_factor([[p for _, p in h] for h in histories.values()])
cuts_pct = []
for h in histories.values():
    prices = [p for _, p in h]
    if len(prices) >= 2 and prices[0] > 0:
        cuts_pct.append(100 * (prices[0] - prices[-1]) / prices[0])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

if changed:
    for aid, h in changed.items():
        ts = [t for t, _ in h]
        ps = [p for _, p in h]
        ax1.plot(ts, ps, marker="o", ms=4, alpha=0.7)
    ax1.set_title(f"Adverts that changed price (n={len(changed)})")
    ax1.set_xlabel("Observed at")
    ax1.set_ylabel("Asking price")
    ax1.yaxis.set_major_formatter(EUR)
    fig.autofmt_xdate()
else:
    ax1.text(0.5, 0.5, "No price changes recorded yet", ha="center", va="center",
             transform=ax1.transAxes, color="gray")
    ax1.set_title("Adverts that changed price")

if cuts_pct:
    ax2.hist(cuts_pct, bins=min(20, max(5, len(cuts_pct))), color="teal",
             alpha=0.75, edgecolor="white")
    med = cut_signal.median_cut
    if med is not None:
        ax2.axvline(100 * med, color="crimson", lw=2,
                    label=f"median cut {100*med:.1f}%  (n={cut_signal.n})")
        ax2.legend()
    ax2.set_title("Distribution of first\u2192last cut %")
    ax2.set_xlabel("Price cut (%)")
    ax2.set_ylabel("Adverts")
else:
    ax2.text(0.5, 0.5, "No multi-observation adverts yet", ha="center", va="center",
             transform=ax2.transAxes, color="gray")
    ax2.set_title("Distribution of first\u2192last cut %")

plt.tight_layout()
plt.show()


## 5 · Days-on-market vs. price percentile

For **delisted** (sold-proxy) adverts: how cheap they were relative to the
fitted curve (residual percentile — left = underpriced) against how long they
lasted. The expectation is that underpricing buys speed.

In [ ]:
fig, ax = plt.subplots()
sold = [r for r in usable if not r.is_active and r.days_on_market is not None]

pts = []
if curve is not None and sold:
    for r in sold:
        pred = analysis.predict(curve, r.year, r.mileage_km, r.fuel_type, r.gearbox)
        resid = math.log(r.price) - math.log(pred.estimate)  # <0 = cheaper than the curve
        pts.append((resid, r.days_on_market))

if pts:
    resids = np.array([p[0] for p in pts])
    doms = np.array([p[1] for p in pts])
    # Residual -> percentile rank within the sold set (0 = cheapest vs curve).
    order = resids.argsort()
    pct = np.empty_like(resids)
    pct[order] = np.linspace(0, 100, len(resids))
    ax.scatter(pct, doms, s=50, alpha=0.8, color="darkorange", edgecolor="white")
    if len(pts) >= 3:
        b, a = np.polyfit(pct, doms, 1)
        xs = np.array([0, 100])
        ax.plot(xs, a + b * xs, color="crimson", lw=2,
                label=f"trend {b:+.2f} days / pctile")
        ax.legend()
    ax.set_xlabel("Price percentile vs. fitted curve  (0 = most underpriced)")
    ax.set_ylabel("Days on market")
    ax.set_title(f"{MODEL_LABEL} \u2014 underpricing vs. speed (delisted, n={len(pts)})")
else:
    ax.text(0.5, 0.5, "Not enough delisted adverts with a fitted curve yet",
            ha="center", va="center", transform=ax.transAxes, color="gray")
    ax.set_title(f"{MODEL_LABEL} \u2014 underpricing vs. speed")
plt.show()


## 6 · Market state over time

Active-listing count reconstructed from advert enter (`first_seen_at`) / leave
(`delisted_at`) events, and the median asking price by week from
`PriceObservation`. Both thicken as daily runs accrue — expect a short window
until then.

In [ ]:
# Enter/leave events -> a running active count over time.
# Same scope as `records` above (make/model *and* the fuel filter).
scoped_ids = {r.ad_id for r in records}
listings = [l for l in db.all_listings() if l.ad_id in scoped_ids]

events = []
for l in listings:
    if l.first_seen_at is not None:
        events.append((pd.Timestamp(l.first_seen_at), +1))
    if l.delisted_at is not None:
        events.append((pd.Timestamp(l.delisted_at), -1))

# Weekly median asking price from the price-observation log.
obs = [(pd.Timestamp(t), p) for h in histories.values() for t, p in h]

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
plotted = False

if events:
    ev = pd.DataFrame(events, columns=["ts", "delta"]).sort_values("ts")
    ev["active"] = ev["delta"].cumsum()
    ax1.step(ev.ts, ev.active, where="post", color="steelblue", lw=2, label="active listings")
    ax1.set_ylabel("Active listings", color="steelblue")
    ax1.tick_params(axis="y", labelcolor="steelblue")
    plotted = True

if obs:
    ob = pd.DataFrame(obs, columns=["ts", "price"]).set_index("ts").sort_index()
    weekly = ob["price"].resample("W").median().dropna()
    if not weekly.empty:
        ax2.plot(weekly.index, weekly.values, color="crimson", marker="o", lw=2,
                 label="median price / week")
        ax2.yaxis.set_major_formatter(EUR)
        ax2.set_ylabel("Median asking price", color="crimson")
        ax2.tick_params(axis="y", labelcolor="crimson")
        plotted = True

if plotted:
    fig.autofmt_xdate()
    ax1.set_xlabel("Date")
    ax1.set_title(f"{MODEL_LABEL} \u2014 market state over time")
else:
    ax1.text(0.5, 0.5, "No dated history yet", ha="center", va="center",
             transform=ax1.transAxes, color="gray")
plt.show()


---
*Estimation methodology and the asking→sale adjustment live in `analysis.py`
(Part B). This notebook is a thin visual layer over it — see `PRICING_PLAN.md`
Parts B–C.*